# Emotion Detection

Given an image of a facial expression, we want to determine what emotion is being portrayed. This is a multiclass classification problem as there are a total of 7 different emotions that we can classify instances as: angry, disgust, fear, happy, sad, surprised, and neutral. Our goal is to build and compare different Convolutional Neural Network (CNN) models to classify these emotions from inputted images.

## Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.utils import to_categorical
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay, classification_report
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, BatchNormalization, MaxPooling2D, Dropout, Flatten, Dense
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import pickle

## Load Preprocessed Data

In [ ]:
data = np.load('../data/processed_data.npz')

X_train = data['X_train']
X_test = data['X_test']
X_val = data['X_val']

X_train_normalized = data['X_train_normalized']
X_test_normalized = data['X_test_normalized']
X_val_normalized = data['X_val_normalized']

y_train = data['y_train']
y_test = data['y_test']
y_val = data['y_val']

y_train_cat = data['y_train_cat']
y_test_cat = data['y_test_cat']
y_val_cat = data['y_val_cat']

## Data Augmentation

There is a significant difference in the number of images across the different emotion classes. This could potentially lead to overfitting as our model will be trained using # layer that randomly applies transformations (flips, rotations, zooms) to each input image during training only
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])more images from certain classes than others. In order to get more data to use to train our model and to fix the imbalance, we will oversample the training data by performing image augmentation. We choose to oversample instead of undersample as undersampling can lead to the loss of important data. We also do not want to duplicate images as this can also lead to overfitting.

Perform data augmentation on all classes (not just minority ones) to create new versions of existing images. This teaches the model to handle small variations in images.

In [ ]:
# layer that randomly applies transformations (flips, rotations, zooms) to each input image during training only
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
])

## Class Weights

Use class weights to tell the loss function to pay more attention to less represented classes (like disgust and fear).

In [ ]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights_dict = dict(enumerate(class_weights))

## Helper Functions

In [ ]:
# plots accuracy and loss training curves
def plot_training_curves(history, title):
    plt.figure(figsize=(12, 5))

    # accuracy
    plt.subplot(1, 2, 1)
    plt.plot(history['accuracy'], label='train')
    plt.plot(history['val_accuracy'], label='val')
    plt.title(f'{title} - Accuracy')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy');
    plt.legend();
    
    # loss
    plt.subplot(1, 2, 2)
    plt.plot(history['loss'], label='train')
    plt.plot(history['val_loss'], label='val')
    plt.title(f'{title} - Loss')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend();

    plt.show()

emotion_labels = ['Angry', 'Disgust', 'Fear', 'Happy', 'Sad', 'Surprise', 'Neutral']

# evaluates model, test accuracy & loss, confusion matrix, classification report
def evaluate_model(model, X_test, y_test, title):
    # test results
    test_accuracy, test_loss = model.evaluate(X_test, y_test, verbose=0)
    print(f'{title} - Test Accuracy: {test_accuracy:.4f} Test Loss: {test_loss:.4f}')
    
    # confusion matrix
    y_pred = np.argmax(model.predict(X_test), axis=1)
    y_true = np.argmax(y_test, axis=1)
    cm = confusion_matrix(y_true, y_pred)
    ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=emotion_labels).plot(cmap='Blues', xticks_rotation='vertical')
    plt.title(f'{title} - Confusion Matrix')
    plt.show()
    
    # classification report
    print(classification_report(y_true, y_pred, target_names=emotion_labels))
    

## Build Models


Build a Convolutional Neural Network (CNN) that classifies images by 1 of 7 emotion types.

### Model 1 (Baseline)

Start by creating a simple model with only 1 convolutional layer.

In [ ]:
model1 = tf.keras.Sequential([
    # define input shape, 48x48 pixels, 1 channel (grayscale)
    layers.Input(shape=(48, 48, 1)),

    # convolutional layer
    layers.Conv2D(32, (3, 3), activation='relu'),

    # convert 2D feature map into a 1D vector
    layers.Flatten(),

    # fully connected layer that learns combinations of the features detected by the convolutional layer, 64 neurons = 64 combinations/patterns
    layers.Dense(64, activation='relu'),

    # final layer with 7 neurons (one for each emotion)
    layers.Dense(7, activation='softmax')
])

model1.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history1 = model1.fit(
    X_train_normalized, y_train_cat,
    epochs=30,
    batch_size=64,
    validation_data=(X_val_normalized, y_val_cat),
    callbacks=[early_stop]
)

# save history during training
with open('../history/history1.pkl', 'wb') as f:
    pickle.dump(history1.history, f)

model1.save('../models/model_v1.keras')

In [ ]:
plot_training_curves(history1, "Model 1")
evaluate_model(model1, X_test_normalized, y_test_cat, "Model 1")

Model 1 is overfitting, it does well on training data but poorly on validation data. Model is too simple/shallow, only 1 convolutional layer and 1 dense layer is not enough to learn and capture complexity of facial features.

The happy emotion was the easiest class to detect whereas disgust was completely missed. Since certain emotions have more images than others, we can possibly introduce class weights or perform data augmentation to address the class imbalance.

Overall accuracy = 41%, F1 score = 0.40

### Model 2 (Data Augmentation)

See how applying data augmentation affects the accuracy of the model.

In [ ]:
model2 = tf.keras.Sequential([
    layers.Input(shape=(48, 48, 1)),

    data_augmentation,

    # normalize pixels after augmentation
    layers.Rescaling(1./255),

    layers.Conv2D(32, (3, 3), activation='relu'),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),
    
    layers.Dense(7, activation='softmax')
])

model2.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history2 = model2.fit(
    X_train, y_train_cat,
    epochs=30,
    batch_size=64,
    validation_data=(X_val, y_val_cat),
    callbacks=[early_stop]
)

with open('../history/history2.pkl', 'wb') as f:
    pickle.dump(history2.history, f)

model2.save('../models/model_v2.keras')

In [ ]:
plot_training_curves(history2, "Model 2")
evaluate_model(model2, X_test, y_test_cat, "Model 2")

Model 2 training and validation accuracies are close so only slightly overfitting. Adding data augmentation helped improve on the model but not by much as the digust class was still completely missed.

Overall accuracy = 43%, F1 score = 0.42

### Model 3 (Class Weights)

See how applying class weights affects the accuracy of the model.

In [ ]:
model3 = tf.keras.Sequential([
    layers.Input(shape=(48, 48, 1)),

    layers.Rescaling(1./255),

    layers.Conv2D(32, (3, 3), activation='relu'),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),

    layers.Dense(7, activation='softmax')
])

model3.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history3 = model3.fit(
    X_train, y_train_cat,
    epochs=30,
    batch_size=64,
    validation_data=(X_val, y_val_cat),
    class_weight=class_weights_dict,
    callbacks=[early_stop]
)

with open('../history/history3.pkl', 'wb') as f:
    pickle.dump(history3.history, f)

model3.save('../models/model_v3.keras')

In [ ]:
plot_training_curves(history3, "Model 3")
evaluate_model(model3, X_test, y_test_cat, "Model 3")

Model 3 is actually recognizing disgust now, but validation and test accuracy dropped slightly. Trade overall accuracy for fairer class performance.

### Model 4 (Data Augmentation + Class Weights)

Try applying both data augmentation and class weights.

In [ ]:
model4 = tf.keras.Sequential([
    layers.Input(shape=(48, 48, 1)),

    data_augmentation,
    layers.Rescaling(1./255),

    layers.Conv2D(32, (3, 3), activation='relu'),

    layers.Flatten(),
    layers.Dense(64, activation='relu'),

    layers.Dense(7, activation='softmax')
])

model4.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history4 = model4.fit(
    X_train, y_train_cat,
    epochs=30,
    batch_size=64,
    validation_data=(X_val, y_val_cat),
    class_weight=class_weights_dict,
    callbacks=[early_stop]
)

with open('../history/history4.pkl', 'wb') as f:
    pickle.dump(history4.history, f)

model4.save('../models/model_v4.keras')

In [ ]:
plot_training_curves(history4, "Model 4")
evaluate_model(model4, X_test, y_test_cat, "Model 4")

Model 4 detects rare emotions more now instead of ignoring them completely. More balanced performance across all emotions. Overall test accuracy dropped but model is more fair now.

### Model 5 - Add More Layers

Apply both data augmentation and class weights to the model. Also try adding more layers to the model.

In [ ]:
model5 = tf.keras.Sequential([
    layers.Input(shape=(48, 48, 1)),

    data_augmentation,
    layers.Rescaling(1./255),

    # 1st layer
    layers.Conv2D(32, (3, 3), activation='relu'),

    # 2nd layer
    layers.Conv2D(64, (3, 3), activation='relu'),

    # 3rd layer
    layers.Conv2D(128, (3, 3), activation='relu'),

    # classifier
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(7, activation='softmax')
])

model5.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history5 = model5.fit(
    X_train, y_train_cat,
    epochs=50,
    batch_size=64,
    validation_data=(X_val, y_val_cat),
    class_weight=class_weights_dict,
    callbacks=[early_stop]
)

with open('../history/history5.pkl', 'wb') as f:
    pickle.dump(history5.history, f)

model5.save('../models/model_v5.keras')

In [ ]:
model5.summary()

plot_training_curves(history5, "Model 5")
evaluate_model(model5, X_test, y_test_cat, "Model 5")

Adding more layers didn't seem to help since model accuracy dropped from 38% to 16%. Model 5 also doesn't detect rarer emotions like disgust and fear well unlike model 4 which was more balanced. This model ended up with too many parameters. To improve we can try adding spatial compression and normalization after each layer.

### Model 6

Add max pooling to reduce spatial dimensions to prevent overfitting and reduce computation time/parameters while still keeping the most important features. Choose a pool size of 2x2 to halve each spatial dimension at each layer. Additionally, apply dropout after the dense layer to prevent overfitting.

In [ ]:
model6 = tf.keras.Sequential([
    layers.Input(shape=(48, 48, 1)),

    data_augmentation,
    layers.Rescaling(1./255),

    # 1st layer
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPool2D((2, 2)),

    # 2nd layer
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPool2D((2, 2)),

    # 3rd layer
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPool2D((2, 2)),

    # classifier
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(7, activation='softmax')
])

model6.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history6 = model6.fit(
    X_train, y_train_cat,
    epochs=50,
    batch_size=64,
    validation_data=(X_val, y_val_cat),
    class_weight=class_weights_dict,
    callbacks=[early_stop]
)

with open('../history/history6.pkl', 'wb') as f:
    pickle.dump(history6.history, f)

model6.save('../models/model_v6.keras')

In [ ]:
model6.summary()
plot_training_curves(history6, "Model 6")
evaluate_model(model6, X_test, y_test_cat, "Model 6")

Model performed much better after introducing regularization and spatial compression. Model accuracy increased from 38% to 47%.

### Model 7

Build on model 6 by doubling number of filters used in each layer.

In [ ]:
model7 = tf.keras.Sequential([
    layers.Input(shape=(48, 48, 1)),

    data_augmentation,
    layers.Rescaling(1./255),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(256, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(7, activation='softmax')
])

model7.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history7 = model7.fit(
    X_train, y_train_cat,
    epochs=50,
    batch_size=64,
    validation_data=(X_val, y_val_cat),
    class_weight=class_weights_dict,
    callbacks=[early_stop]
)

with open('../history/history7.pkl', 'wb') as f:
    pickle.dump(history7.history, f)

model7.save('../models/model_v7.keras')

In [ ]:
model7.summary()
plot_training_curves(history7, "Model 7")
evaluate_model(model7, X_test, y_test_cat, "Model 7")

Validation accuracy reached 44%. Additional filters did not translate to improved accuracy as model 6 performed better overall. Try adding regularization after each layer next to see if it improves the model.

### Model 8

Try adding dropout regularization after each convolutional layer to prevent overfitting.

In [ ]:
model8 = tf.keras.Sequential([
    layers.Input(shape=(48, 48, 1)),

    data_augmentation,
    layers.Rescaling(1./255),

    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.25),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.25),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.25),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(7, activation='softmax')
])

model8.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

history8 = model8.fit(
    X_train, y_train_cat,
    epochs=50,
    batch_size=64,
    validation_data=(X_val, y_val_cat),
    class_weight=class_weights_dict,
    callbacks=[early_stop]
)

with open('..history/history8.pkl', 'wb') as f:
    pickle.dump(history8.history, f)

model8.save('../models/model_v8.keras')

### Model 9

Add convolutional blocks instead of just single layers. Increase training time from 50 epochs to 100.

In [ ]:
model9 = tf.keras.Sequential([
    layers.Input(shape=(48, 48, 1)),

    data_augmentation,
    layers.Rescaling(1./255),

    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(7, activation='softmax')
])

model9.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history9 = model9.fit(
    X_train, y_train_cat,
    epochs=100,
    batch_size=64,
    validation_data=(X_val, y_val_cat),
    class_weight=class_weights_dict,
    callbacks=[early_stop]
)

with open('../history/history9.pkl', 'wb') as f:
    pickle.dump(history9.history, f)

model9.save('../models/model_v9.keras')

### Model 10

Build on model 9 by doubling the number of filters used in each convolutional block.

In [ ]:
model10 = tf.keras.Sequential([
    layers.Input(shape=(48, 48, 1)),

    data_augmentation,
    layers.Rescaling(1./255),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(256, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.Conv2D(256, (3, 3), activation='relu'),
    layers.BatchNormalization(),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(7, activation='softmax')
])

model10.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history10 = model10.fit(
    X_train, y_train_cat,
    epochs=100,
    batch_size=64,
    validation_data=(X_val, y_val_cat),
    class_weight=class_weights_dict,
    callbacks=[early_stop]
)

with open('..history/history10.pkl', 'wb') as f:
    pickle.dump(history10.history, f)

model10.save('../models/model_v10.keras')

Model 10 achieved 60% test accuracy and showed improved generalization compared to the previous models. It performed the best on the happy and surprised emotions but could improve on disgust and fear. Likely due to an imbalanced dataset. Focus on hyperparameter tuning next.